# 9.11 Flipping one batch's orientation in v space

Run: `sigma=2, seed=3, model5, bif, gene-mode, decipher seed 1` from the 0918 sweep. `batch03` sits
in a mirrored frame relative to the other batches in `obsm["decipher_v"]`.

Findings so far: `w_bio` / `w_branch` are shared across batches in the simulator, so the flip is a
gauge choice of the learned `v`, not a property of the data. Procrustes against ground-truth
`latent_v` gives det = -1 for batches 00/01/02/04 and det = +1 for batch03, so batch03 differs by a
**reflection**. `v -> -v` is a 180-degree rotation (det +1) and cannot fix it.

In [ ]:
from pathlib import Path
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import orthogonal_procrustes

def repo_root():
    p = Path.cwd().resolve()
    while not (p / "decipher_models").is_dir():
        if p == p.parent:
            raise FileNotFoundError("run from inside decipher-bc")
        p = p.parent
    return p

ROOT = repo_root()
H5AD = (ROOT / "Simulated Data/Sweeps and Results/shift_sigma_sweep_bifurcation/0918/trained"
        / "sweep0918_ds3_sigma2.0_seed3_model5_bif_genes_decipherseed_1.h5ad")
FLIP = "batch03"
a = ad.read_h5ad(H5AD)
a.obs_names_make_unique()
V0 = a.obsm["decipher_v"].copy()
L = a.obsm["latent_v"]
a

## Per-batch Procrustes fit to ground truth
angle/det of the map decipher_v -> latent_v for each batch.

In [ ]:
def fit(V, L, m):
    R, _ = orthogonal_procrustes(V[m] - V[m].mean(0), L[m] - L[m].mean(0))
    return R

for b in sorted(a.obs.batch.unique()):
    R = fit(V0, L, (a.obs.batch == b).values)
    print(b, "det", round(np.linalg.det(R)), "angle", round(np.degrees(np.arctan2(R[1, 0], R[0, 0]))))

## Flip methods
All take `V` and return a new `V`; only cells of `FLIP` change.

- `procrustes_truth`: map `FLIP` onto the other batches' frame via ground-truth `latent_v` (simulated data only). Recenters on the others' centroid.
- `reflect_axis`: reflect `FLIP` across a chosen axis (0 = flip v1, 1 = flip v2) about its own centroid. No ground truth needed; try both axes.
- `negate`: `v -> -v` about its own centroid (a rotation, kept for comparison).

In [ ]:
def procrustes_truth(V, m):
    V = V.copy(); ref = ~m
    mu_ref, mu_l = V[ref].mean(0), L[ref].mean(0)
    R_ref, _ = orthogonal_procrustes(V[ref] - mu_ref, L[ref] - mu_l)
    mu3 = V[m].mean(0)
    R3 = fit(V, L, m)
    V[m] = (V[m] - mu3) @ R3 @ R_ref.T + mu_ref
    return V

def reflect_axis(V, m, axis=0, recenter=False):
    V = V.copy(); mu = V[m].mean(0)
    S = np.eye(2); S[axis, axis] = -1
    V[m] = (V[m] - mu) @ S + (V[~m].mean(0) if recenter else mu)
    return V

def negate(V, m):
    V = V.copy(); mu = V[m].mean(0)
    V[m] = -(V[m] - mu) + mu
    return V

m = (a.obs.batch == FLIP).values
variants = {
    "original": V0,
    "procrustes_truth": procrustes_truth(V0, m),
    "reflect_axis 0": reflect_axis(V0, m, 0),
    "reflect_axis 1": reflect_axis(V0, m, 1),
    "negate": negate(V0, m),
}

In [ ]:
batches = sorted(a.obs.batch.unique())
colors = dict(zip(batches, plt.cm.tab10.colors))
fig, axes = plt.subplots(2, len(variants), figsize=(4 * len(variants), 8), sharex=False)
for j, (name, V) in enumerate(variants.items()):
    for b in batches:
        k = (a.obs.batch == b).values
        axes[0, j].scatter(*V[k].T, s=3, color=colors[b], label=b)
    sc = axes[1, j].scatter(*V.T, s=3, c=a.obs["latent_t"], cmap="viridis")
    axes[0, j].set_title(name); axes[0, j].set_aspect("equal"); axes[1, j].set_aspect("equal")
axes[0, 0].legend(markerscale=4, fontsize=7)
fig.colorbar(sc, ax=axes[1, -1], label="latent_t")
fig.suptitle(f"decipher_v, flipping {FLIP}")
plt.show()

## Check: does a method make FLIP's frame match the others?
Procrustes det of each batch's map to `latent_v` after the flip should be identical across batches (the others were det -1).

In [ ]:
for name, V in variants.items():
    dets = {b: round(np.linalg.det(fit(V, L, (a.obs.batch == b).values))) for b in batches}
    print(f"{name:18s}", dets)

## Reflect across the line through the two arm tips
Line `y = a x + b` through the mean v of each arm's tip (cells with `latent_t > TIP_Q` on branch +1 / -1), then reflect every `FLIP` cell across it. The tips stay fixed and the trunk swings to the other side, so no rotation step is needed. Uses `branch_id` / `latent_t` (simulated data only) to pick the tips.

The reflection is written in point-plus-direction form so a vertical line doesn't break it; `a`, `b` are printed for reference.

In [ ]:
TIP_Q = 0.9
br = a.obs["branch_id"].values
t = a.obs["latent_t"].values

def arm_tips(V, mask, q=TIP_Q):
    return [V[mask & (br == s) & (t > q)].mean(0) for s in (1, -1)]

def reflect_across_line(V, m, p, q):
    """Reflect rows V[m] across the line through points p and q."""
    V = V.copy()
    d = (q - p) / np.linalg.norm(q - p)
    X = V[m] - p
    V[m] = p + 2 * np.outer(X @ d, d) - X   # projection onto line, doubled, minus the point
    return V

p, q = arm_tips(V0, m)                      # FLIP's own tips
a_, b_ = np.polyfit([p[0], q[0]], [p[1], q[1]], 1)
print(f"tips {np.round(p, 2)} {np.round(q, 2)}   y = {a_:.3f} x + {b_:.3f}")
variants["tip_line_reflect"] = reflect_across_line(V0, m, p, q)

In [ ]:
from scipy.spatial import cKDTree
def overlap(V, m):
    """median distance from each FLIP cell to its nearest cell in the other batches (lower = better)"""
    return np.median(cKDTree(V[~m]).query(V[m])[0])

for name, V in variants.items():
    print(f"{name:18s} {overlap(V, m):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, name in zip(axes, ["original", "tip_line_reflect", "procrustes_truth"]):
    V = variants[name]
    ax.scatter(*V[~m].T, s=3, color="0.75")
    ax.scatter(*V[m].T, s=3, color="C3", label=FLIP)
    ax.plot(*np.c_[p, q], "k--", lw=1)
    ax.set_title(name); ax.set_aspect("equal")
axes[0].legend(markerscale=4)
plt.show()

## Save (optional)
Stores the chosen variant next to the original without touching the h5ad on disk.

In [ ]:
CHOICE = "procrustes_truth"
a.obsm["decipher_v_aligned"] = variants[CHOICE]